# Tiny CNN Walkthrough: notebook

The same network as the interactive page (`index.html`): a 6×6 image, two 3×3 kernels, ReLU, 2×2 max pooling, flatten, a dense layer and softmax, classifying a drawing as a vertical line, horizontal line, cross/box or empty.

Part 1 does the forward pass in NumPy, part 2 does it with TensorFlow, and part 3 trains the weights from random values. Every printed output in parts 1 and 2 matches the interactive page.

In Google Colab everything is preinstalled. Locally, run `pip install -r requirements.txt` first.

## Part 1: forward pass in NumPy
The weights are the hand-set values from the page, so the output is predictable.

### Steps 1–3: convolution, ReLU and max pooling
A 3×3 kernel slides over the 6×6 image (6 − 3 + 1 = 4 positions per axis), ReLU sets negatives to 0, and 2×2 max pooling keeps the largest value of each block.

In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

np.set_printoptions(precision=4, suppress=True)   # at most 4 decimals in printouts

img = np.array([[0, 0, 1, 1, 0, 0],
                [0, 0, 1, 1, 0, 0],
                [0, 0, 1, 1, 0, 0],
                [0, 0, 1, 1, 0, 0],
                [0, 0, 1, 1, 0, 0],
                [0, 0, 1, 1, 0, 0]])
print(img.shape)

(6, 6)


In [2]:
K = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]])   # vertical kernel
windows = sliding_window_view(img, (3, 3))
print(windows.shape)   # one 3×3 patch per position

(4, 4, 3, 3)


In [3]:
# the highlighted window and its weighted sum
print(windows[0, 0])
print((windows[0, 0] * K).sum())

[[0 0 1]
 [0 0 1]
 [0 0 1]]
-3


In [4]:
conv = (windows * K).sum(axis=(2, 3))
print(conv)

[[-3 -3  3  3]
 [-3 -3  3  3]
 [-3 -3  3  3]
 [-3 -3  3  3]]


In [5]:
act = np.maximum(0, conv)   # ReLU
print(act)

[[0 0 3 3]
 [0 0 3 3]
 [0 0 3 3]
 [0 0 3 3]]


In [6]:
# split rows and columns into blocks of 2, keep each block's max
pool = act.reshape(2, 2, 2, 2).max(axis=(1, 3))
print(pool)

[[0 3]
 [0 3]]


### Step 4: flatten
Both kernels run on the same image; their pooled 2×2 maps are laid end to end as one vector of 8 numbers.

In [7]:
def features(K):
    conv = (sliding_window_view(img, (3, 3)) * K).sum(axis=(2, 3))
    act = np.maximum(0, conv)
    return act.reshape(2, 2, 2, 2).max(axis=(1, 3))

KV = np.array([[1, 0, -1],
               [1, 0, -1],
               [1, 0, -1]])
KH = np.array([[1, 1, 1],
               [0, 0, 0],
               [-1, -1, -1]])

pool_v, pool_h = features(KV), features(KH)
print(pool_v.ravel(), pool_h.ravel())

[0 3 0 3] [0 0 0 0]


In [8]:
x = np.concatenate([pool_v.ravel(), pool_h.ravel()])
print(x.shape)
print(x)

(8,)
[0 3 0 3 0 0 0 0]


### Step 5: dense layer
One row of 8 weights plus a bias per class. Each logit is the dot product of its row with the flattened vector, plus the bias.

In [9]:
W = np.array([[1, 1, 1, 1, -1, -1, -1, -1],
              [-1, -1, -1, -1, 1, 1, 1, 1],
              [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
              [-0.5, -0.5, -0.5, -0.5, -0.5, -0.5, -0.5, -0.5]])   # (4, 8): one row per class
b = np.array([0, 0, -3, 2])

logits = W @ x + b
print(logits)

[ 6. -6.  0. -1.]


In [10]:
# one row on its own: "Vertical line"
print(round(float(W[0] @ x + b[0]), 4))

6.0


### Step 6: softmax prediction
Softmax turns the logits into probabilities; the highest one is the prediction.

In [11]:
p = np.exp(logits - logits.max())   # subtract max for numerical stability
p = p / p.sum()
print(p.round(3))

[0.997 0.    0.002 0.001]


In [12]:
CLASSES = ["Vertical line", "Horizontal line", "Cross / box", "Empty"]
print(CLASSES[p.argmax()])

Vertical line


## Part 2: the same forward pass with TensorFlow
`tf.nn.conv2d`, `tf.nn.relu` and `tf.nn.max_pool2d` on a `(1, 6, 6, 1)` tensor, then a real Keras `Dense` layer. TensorFlow computes in float32, so the printouts are limited to 4 decimals.

### Steps 1–3: convolution, ReLU and max pooling

In [13]:
import numpy as np
import tensorflow as tf

np.set_printoptions(precision=4, suppress=True)   # hides float32 noise such as 0.90000004

img = tf.constant([[0, 0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0, 0]], dtype=tf.float32)
x_in = tf.reshape(img, (1, 6, 6, 1))   # batch, height, width, channels
print(x_in.shape)

(1, 6, 6, 1)


In [14]:
K = tf.constant([[1, 0, -1],
                 [1, 0, -1],
                 [1, 0, -1]], dtype=tf.float32)   # vertical kernel
filters = tf.reshape(K, (3, 3, 1, 1))   # height, width, in channels, out channels
print(filters.shape)

(3, 3, 1, 1)


In [15]:
# the highlighted window and its weighted sum
window = img[0:3, 0:3]
print(window.numpy())
print(round(float(tf.reduce_sum(window * K)), 4))

[[0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]]
-3.0


In [16]:
conv = tf.nn.conv2d(x_in, filters, strides=1, padding="VALID")
print(conv.shape)
print(conv[0, :, :, 0].numpy())

(1, 4, 4, 1)
[[-3. -3.  3.  3.]
 [-3. -3.  3.  3.]
 [-3. -3.  3.  3.]
 [-3. -3.  3.  3.]]


In [17]:
act = tf.nn.relu(conv)
print(act[0, :, :, 0].numpy())

[[0. 0. 3. 3.]
 [0. 0. 3. 3.]
 [0. 0. 3. 3.]
 [0. 0. 3. 3.]]


In [18]:
pool = tf.nn.max_pool2d(act, ksize=2, strides=2, padding="VALID")
print(pool.shape)
print(pool[0, :, :, 0].numpy())

(1, 2, 2, 1)
[[0. 3.]
 [0. 3.]]


### Step 4: flatten

In [19]:
KV = tf.constant([[1, 0, -1],
                  [1, 0, -1],
                  [1, 0, -1]], dtype=tf.float32)
KH = tf.constant([[1, 1, 1],
                  [0, 0, 0],
                  [-1, -1, -1]], dtype=tf.float32)
filters = tf.stack([KV, KH], axis=-1)[:, :, tf.newaxis, :]   # (3, 3, 1, 2)

conv = tf.nn.conv2d(x_in, filters, strides=1, padding="VALID")
act = tf.nn.relu(conv)
pool = tf.nn.max_pool2d(act, ksize=2, strides=2, padding="VALID")
print(pool.shape)   # 2 feature maps of 2×2

(1, 2, 2, 2)


In [20]:
# channels first, so the order matches the page: x0-x3 vertical, x4-x7 horizontal
x = tf.keras.layers.Flatten()(tf.transpose(pool, [0, 3, 1, 2]))
print(x.shape)
print(x.numpy())

(1, 8)
[[0. 3. 0. 3. 0. 0. 0. 0.]]


In [21]:
# for comparison: plain Flatten() on channels-last data interleaves the maps
print(tf.keras.layers.Flatten()(pool).numpy())

[[0. 0. 3. 0. 0. 0. 3. 0.]]


### Step 5: dense layer

In [22]:
W = tf.constant([[1, 1, 1, 1, -1, -1, -1, -1],
                 [-1, -1, -1, -1, 1, 1, 1, 1],
                 [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
                 [-0.5, -0.5, -0.5, -0.5, -0.5, -0.5, -0.5, -0.5]], dtype=tf.float32)   # (4, 8): one row per class
b = tf.constant([0, 0, -3, 2], dtype=tf.float32)

dense = tf.keras.layers.Dense(4)
dense.build((None, 8))
dense.set_weights([tf.transpose(W).numpy(), b.numpy()])   # Keras stores the kernel as (8, 4)
logits = dense(x)
print(logits.numpy())

[[ 6. -6.  0. -1.]]


In [23]:
# one row on its own: "Vertical line"
print(round(float(tf.tensordot(W[0], x[0], axes=1) + b[0]), 4))

6.0


### Step 6: softmax prediction

In [24]:
p = tf.nn.softmax(logits)[0]
print(p.numpy().round(3))

[0.997 0.    0.002 0.001]


In [25]:
CLASSES = ["Vertical line", "Horizontal line", "Cross / box", "Empty"]
print(CLASSES[int(tf.argmax(p))])

Vertical line


## Part 3: train the weights from scratch
Starting from random weights, mini-batch gradient descent with hand-written backpropagation. The numbers differ from run to run; accuracy normally climbs to 0.9–1.0. The page's Train mode does the same thing interactively and also offers SGD + momentum and Adam.

In [26]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

def make_dataset():
    X, y = [], []
    for w in (1, 2):                                   # vertical and horizontal lines
        for c in range(7 - w):
            v = np.zeros((6, 6)); v[:, c:c + w] = 1
            X += [v, v.T.copy()]; y += [0, 1]
    crosses = [(c, r, 2) for c in (1, 2, 3) for r in (1, 2, 3)] + [(1, 1, 1), (3, 2, 1), (4, 4, 1)]
    for c, r, w in crosses:
        m = np.zeros((6, 6)); m[:, c:c + w] = 1; m[r:r + w, :] = 1
        X.append(m); y.append(2)
    for s in (4, 5, 6):                                # square outlines
        for r in range(7 - s):
            for c in range(7 - s):
                m = np.zeros((6, 6)); m[r:r + s, c:c + s] = 1; m[r + 1:r + s - 1, c + 1:c + s - 1] = 0
                X.append(m); y.append(2)
    X += [np.zeros((6, 6))] * 6; y += [3] * 6          # empty images
    return np.array(X), np.array(y)   # 54 images, labels 0-3

def forward(img, KV, KH, W, b):
    pools, cache = [], []
    for K in (KV, KH):
        conv = (sliding_window_view(img, (3, 3)) * K).sum(axis=(2, 3))
        act = np.maximum(0, conv)                      # ReLU
        blocks = act.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)   # the four 2×2 blocks
        pools.append(blocks.max(axis=1)); cache.append((conv, blocks.argmax(axis=1)))
    x = np.concatenate(pools)
    z = W @ x + b
    p = np.exp(z - z.max()); p /= p.sum()
    return x, p, cache

def gradients(img, label, KV, KH, W, b):
    x, p, cache = forward(img, KV, KH, W, b)
    dz = p.copy(); dz[label] -= 1                      # softmax + cross-entropy: p − y
    dW, db = np.outer(dz, x), dz
    dx = W.T @ dz
    windows = sliding_window_view(img, (3, 3))
    dK = []
    for c, (conv, arg) in enumerate(cache):
        dblocks = np.zeros((4, 4))
        dblocks[np.arange(4), arg] = dx[4 * c:4 * c + 4]   # only each block's max gets gradient
        dconv = dblocks.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3).reshape(4, 4)
        dconv *= conv > 0                                   # ReLU passes gradient only where active
        dK.append((windows * dconv[:, :, None, None]).sum(axis=(0, 1)))
    return -np.log(p[label]), dW, db, dK

X, y = make_dataset()
rng = np.random.default_rng(0)
lim_k, lim_d = np.sqrt(6 / 27), np.sqrt(6 / 12)       # Glorot uniform, as Keras uses
KV, KH = rng.uniform(-lim_k, lim_k, (2, 3, 3))
W, b = rng.uniform(-lim_d, lim_d, (4, 8)), np.zeros(4)
lr, epochs, batch = 0.3, 100, 8

def update(w, g, state):          # plain SGD
    return w - lr * g

state = {name: {} for name in ("W", "b", "KV", "KH")}   # optimizer memory per parameter
step = 0
for epoch in range(epochs):
    order = rng.permutation(len(X))
    for s in range(0, len(X), batch):
        idx = order[s:s + batch]
        g = [gradients(X[i], y[i], KV, KH, W, b) for i in idx]
        step += 1
        W = update(W, np.mean([gi[1] for gi in g], axis=0), state["W"])
        b = update(b, np.mean([gi[2] for gi in g], axis=0), state["b"])
        KV = update(KV, np.mean([gi[3][0] for gi in g], axis=0), state["KV"])
        KH = update(KH, np.mean([gi[3][1] for gi in g], axis=0), state["KH"])
    if epoch % 20 == 0 or epoch == epochs - 1:
        loss = np.mean([gradients(X[i], y[i], KV, KH, W, b)[0] for i in range(len(X))])
        acc = np.mean([forward(X[i], KV, KH, W, b)[1].argmax() == y[i] for i in range(len(X))])
        print(f"epoch {epoch:3d}  loss {loss:.3f}  accuracy {acc:.2f}")

print(KV.round(2), KH.round(2), W.round(2), b.round(2), sep="\n")


epoch   0  loss 0.995  accuracy 0.56


epoch  20  loss 0.228  accuracy 0.94

epoch  40  loss 0.062  accuracy 1.00

epoch  60  loss 0.032  accuracy 1.00


epoch  80  loss 0.021  accuracy 1.00


epoch  99  loss 0.016  accuracy 1.00
[[-1.6  -1.98 -2.14]
 [ 0.05  0.66  0.82]
 [ 1.54  1.32  1.32]]
[[ 2.72 -0.45 -0.32]
 [ 1.64 -1.61  0.78]
 [-0.49 -0.16  2.89]]
[[-1.74 -1.36 -1.49 -1.23 -0.01  0.06  1.63  1.95]
 [ 1.8   1.5   1.44  1.56  1.64 -0.1  -2.11 -1.29]
 [ 0.6   0.35  0.02  0.3   0.37  1.27  0.84  0.28]
 [-0.07 -0.42 -0.56  0.15 -1.26 -1.21 -1.2  -0.57]]
[-0.53  0.13 -3.13  3.53]
